In [ ]:
# Standard library imports
import json
from pathlib import Path
from typing import Optional, Tuple
import warnings

# Third-party imports
import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import box
import folium
from folium import plugins
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, Button, Output, VBox, HBox
from IPython.display import display, HTML

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore')

# Set up matplotlib for inline plotting
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All libraries imported successfully!")
print(f"\nLibrary Versions:")
print(f"  GeoPandas: {gpd.__version__}")
print(f"  Pandas: {pd.__version__}")
print(f"  Folium: {folium.__version__}")

In [ ]:
def create_town_map(town_name: str = None) -> folium.Map:
    """
    Create an interactive map showing a Vermont town with geology layer.
    
    Parameters
    ----------
    town_name : str, optional
        Name of town to display. If None, uses globally selected town.
    
    Returns
    -------
    folium.Map
        Interactive Folium map centered on the town
    
    Notes
    -----
    - Converts town boundary from Vermont State Plane to WGS84 for web display
    - Adds geology tile service from VT ANR
    - Adds town boundary as styled GeoJSON overlay
    - Includes layer controls for toggling visibility
    """
    # Use provided town name or fall back to global selection
    if town_name is None:
        if selected_town_name is None:
            print("⚠️  No town selected. Please select a town first.")
            return None
        town_name = selected_town_name
    
    # Get the town data
    town_data = towns_gdf[towns_gdf[town_name_field] == town_name].copy()
    
    if len(town_data) == 0:
        print(f"⚠️  No data found for town: {town_name}")
        return None
    
    # Convert town boundary to WGS84 for web mapping
    town_wgs84 = town_data.to_crs(WGS84)
    
    # Get the centroid for map center
    centroid = town_wgs84.geometry.centroid.iloc[0]
    center_lat = centroid.y
    center_lon = centroid.x
    
    # Calculate appropriate zoom level based on town size
    bounds = town_wgs84.total_bounds  # [minx, miny, maxx, maxy]
    lat_range = bounds[3] - bounds[1]
    lon_range = bounds[2] - bounds[0]
    max_range = max(lat_range, lon_range)
    
    # Heuristic for zoom level (larger towns need lower zoom)
    if max_range > 0.5:
        zoom_start = 10
    elif max_range > 0.2:
        zoom_start = 11
    else:
        zoom_start = 12
    
    # Create the map
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=zoom_start,
        tiles='OpenStreetMap',
        control_scale=True
    )
    
    # Add alternative basemaps
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Satellite Imagery',
        overlay=False,
        control=True
    ).add_to(m)
    
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Topographic',
        overlay=False,
        control=True
    ).add_to(m)
    
    # Add VT ANR Bedrock Geology as tile layer
    geology_tile_url = f"{GEOLOGY_MAPSERVICE_URL}/tile/{{z}}/{{y}}/{{x}}"
    
    folium.TileLayer(
        tiles=geology_tile_url,
        attr='Vermont Agency of Natural Resources',
        name='Bedrock Geology',
        overlay=True,
        control=True,
        opacity=0.7
    ).add_to(m)
    
    # Style function for town boundary
    def style_function(feature):
        return {
            'fillColor': 'transparent',
            'color': '#ff0000',
            'weight': 3,
            'opacity': 0.8,
            'fillOpacity': 0
        }
    
    # Add town boundary as GeoJSON overlay
    folium.GeoJson(
        town_wgs84,
        name=f'{town_name} Boundary',
        style_function=style_function,
        tooltip=folium.Tooltip(f'<b>{town_name}</b>')
    ).add_to(m)
    
    # Add layer control
    folium.LayerControl(position='topright', collapsed=False).add_to(m)
    
    # Add a marker at the centroid
    folium.Marker(
        [center_lat, center_lon],
        popup=f'<b>{town_name}</b><br>Town Center',
        icon=folium.Icon(color='red', icon='info-sign')
    ).add_to(m)
    
    print(f"✓ Map created for {town_name}")
    print(f"  Center: ({center_lat:.4f}, {center_lon:.4f})")
    print(f"  Zoom Level: {zoom_start}")
    
    return m

In [ ]:
# Create data directory if it doesn't exist
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"✓ Data directory ready: {DATA_DIR.absolute()}")

In [ ]:
# API Endpoints
TOWN_BOUNDARIES_URL = (
    "https://services1.arcgis.com/BkFxaEFNwHqX3tAw/arcgis/rest/services/"
    "FS_VCGI_OPENDATA_Boundary_BNDHASH_poly_towns_SP_v1/FeatureServer/0/query"
    "?outFields=*&where=1%3D1&f=geojson"
)

GEOLOGY_MAPSERVICE_URL = (
    "https://anrmaps.vermont.gov/arcgis/rest/services/Open_Data/"
    "OPENDATA_ANR_GEOLOGIC_SP_NOCACHE_v2/MapServer/165"
)

GEOLOGY_QUERY_ENDPOINT = f"{GEOLOGY_MAPSERVICE_URL}/query"

BASEMAP_URL = (
    "https://basemaps.arcgis.com/arcgis/rest/services/"
    "World_Basemap_v2/VectorTileServer"
)

# Coordinate Reference Systems
VT_STATE_PLANE = "EPSG:32145"  # Vermont State Plane NAD83 (meters)
WEB_MERCATOR = "EPSG:3857"      # Web Mercator (for tile services)
WGS84 = "EPSG:4326"             # WGS84 (latitude/longitude)

# File paths for cached data
TOWNS_CACHE = DATA_DIR / "towns.geojson"

print("✓ Configuration complete!")
print(f"\nData Sources:")
print(f"  Towns: VCGI OpenData Portal")
print(f"  Geology: VT Agency of Natural Resources")
print(f"\nCoordinate Systems:")
print(f"  Vermont State Plane: {VT_STATE_PLANE}")
print(f"  WGS84 (Web): {WGS84}")

In [ ]:
# define a function to fetch town boundaries
def fetch_town_boundaries(use_cache: bool = True) -> gpd.GeoDataFrame:
    """
    Fetch Vermont town boundaries from VCGI OpenData portal.
    
    Parameters
    ----------
    use_cache : bool, default True
        If True, load from local cache if available. Otherwise, fetch from API.
    
    Returns
    -------
    gpd.GeoDataFrame
        GeoDataFrame containing Vermont town boundaries with attributes
    
    Notes
    -----
    Data is cached to data/towns.geojson after first download.
    """
    # Check if cached file exists and use_cache is True
    if use_cache and TOWNS_CACHE.exists():
        print(f"📁 Loading towns from cache: {TOWNS_CACHE}")
        gdf = gpd.read_file(TOWNS_CACHE)
        print(f"✓ Loaded {len(gdf)} towns from cache")
        return gdf
    
    # Fetch from API
    print(f"🌐 Fetching town boundaries from VCGI...")
    print(f"   URL: {TOWN_BOUNDARIES_URL[:80]}...")
    
    try:
        # Make HTTP GET request
        response = requests.get(TOWN_BOUNDARIES_URL, timeout=30)
        response.raise_for_status()  # Raise exception for bad status codes
        
        # Parse GeoJSON response
        geojson_data = response.json()
        
        # Convert to GeoDataFrame
        gdf = gpd.GeoDataFrame.from_features(geojson_data['features'])
        
        # Set coordinate reference system
        # VCGI data is in Vermont State Plane (EPSG:32145)
        gdf.set_crs(VT_STATE_PLANE, inplace=True)
        
        print(f"✓ Fetched {len(gdf)} town boundaries")
        
        # Save to cache
        print(f"💾 Saving to cache: {TOWNS_CACHE}")
        gdf.to_file(TOWNS_CACHE, driver='GeoJSON')
        print(f"✓ Cache saved successfully")
        
        return gdf
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching data: {e}")
        raise
    except Exception as e:
        print(f"❌ Error processing data: {e}")
        raise



In [ ]:
def on_town_selected(town_name: str) -> None:
    """
    Callback function when a town is selected from the dropdown.
    
    Parameters
    ----------
    town_name : str
        Name of the selected town
    """
    global selected_town_data, selected_town_name
    
    # Store the selected town name
    selected_town_name = town_name
    
    # Filter the GeoDataFrame to get the selected town
    selected_town_data = towns_gdf[towns_gdf[town_name_field] == town_name].copy()
    
    if len(selected_town_data) == 0:
        print(f"⚠️  No data found for town: {town_name}")
        return
    
    # Get the first (and should be only) row
    town = selected_town_data.iloc[0]
    
    # Display town information
    print("=" * 70)
    print(f"📍 SELECTED TOWN: {town_name}")
    print("=" * 70)
    
    # Display key attributes
    print(f"\nTown Information:")
    for col in selected_town_data.columns:
        if col != 'geometry':
            value = town[col]
            print(f"  • {col}: {value}")
    
    # Calculate and display area
    # Area is calculated in square meters (since CRS is in meters)
    area_sq_m = town.geometry.area
    area_sq_km = area_sq_m / 1_000_000
    area_acres = area_sq_m / 4046.86
    
    print(f"\nGeographic Properties:")
    print(f"  • Area: {area_sq_km:.2f} km² ({area_acres:.2f} acres)")
    
    # Get bounding box
    bounds = town.geometry.bounds
    print(f"  • Bounding Box (Vermont State Plane, meters):")
    print(f"      Min X: {bounds[0]:,.2f}")
    print(f"      Min Y: {bounds[1]:,.2f}")
    print(f"      Max X: {bounds[2]:,.2f}")
    print(f"      Max Y: {bounds[3]:,.2f}")
    
    # Calculate centroid
    centroid = town.geometry.centroid
    print(f"  • Centroid:")
    print(f"      X: {centroid.x:,.2f}")
    print(f"      Y: {centroid.y:,.2f}")
    
    print("\n" + "=" * 70)
    print("✓ Town data loaded and ready for mapping")
    print("=" * 70)

In [ ]:
# Fetch the data
towns_gdf = fetch_town_boundaries()

print(f"\n" + "="*50)
print("TOWN BOUNDARIES LOADED SUCCESSFULLY")
print("="*50)

In [ ]:
# Identify the town name column
# Common field names: TOWNNAME, TOWN, NAME, etc.
name_candidates = ['TOWNNAME', 'TOWN', 'NAME', 'Town', 'name']
town_name_field = None

town_name_field = "TOWNNAMEMC"

# Get sorted list of town names
town_names = sorted(towns_gdf[town_name_field].unique())

print(f"\n✓ Found {len(town_names)} Vermont towns")
print(f"\nTown name field: '{town_name_field}'")
print(f"\nSample towns (first 10):")
for name in town_names[:10]:
    print(f"  • {name}")
print(f"  ...")
print(f"  • {town_names[-1]}")

In [ ]:
# Display first few towns
# Drop geometry column for cleaner display (it's very long)
display_cols = [col for col in towns_gdf.columns if col != 'geometry']

print("\nFirst 5 Towns:")
print("=" * 60)
towns_gdf[display_cols].head()

In [ ]:
# Display column names and types
print("Available Columns:")
print("=" * 60)
for col in towns_gdf.columns:
    dtype = towns_gdf[col].dtype
    if col != 'geometry':
        sample = towns_gdf[col].iloc[0] if len(towns_gdf) > 0 else None
        print(f"  {col:20s} ({dtype}) - Example: {sample}")
    else:
        print(f"  {col:20s} ({dtype})")

In [ ]:
# Display basic information
print("Dataset Shape:")
print(f"  Rows (towns): {len(towns_gdf)}")
print(f"  Columns (attributes): {len(towns_gdf.columns)}")

print(f"\nCoordinate Reference System:")
print(f"  {towns_gdf.crs}")
print(f"  Name: {towns_gdf.crs.name}")

print(f"\nGeometry Type:")
print(f"  {towns_gdf.geometry.type.unique()}")

print(f"\nBounding Box (in meters, Vermont State Plane):")
bounds = towns_gdf.total_bounds
print(f"  Min X: {bounds[0]:,.2f}")
print(f"  Min Y: {bounds[1]:,.2f}")
print(f"  Max X: {bounds[2]:,.2f}")
print(f"  Max Y: {bounds[3]:,.2f}")

In [ ]:
# Create output widget to capture display
output = Output()

print("🎛️  Interactive Town Selector")
print("=" * 70)
print("Select a Vermont town from the dropdown below to view its information.")
print("This will load the town's boundary data for mapping in the next step.")
print("=" * 70)

# Create the dropdown widget
town_dropdown = Dropdown(
    options=town_names,
    value=town_names[0],  # Default to first town
    description='Select Town:',
    style={'description_width': '100px'},
    layout={'width': '400px'}
)

# Wrapper function to clear output before displaying
def update_town_display(town_name):
    with output:
        output.clear_output(wait=True)
        on_town_selected(town_name)

# Use Dropdown.observe instead of interact to avoid duplicate callbacks
def _on_dropdown_change(change):
    # Only respond to user-driven value changes
    if change.get('name') == 'value' and change.get('new') is not None:
        update_town_display(change.get('new'))

# Attach the observer
town_dropdown.observe(_on_dropdown_change, names='value')

# Display the dropdown and the output widget
display(town_dropdown)
display(output)

# The observer will now handle the initial population of the output
# when the dropdown is displayed.

In [ ]:
# Create and display the map for the selected town
# Make sure you've selected a town using the dropdown above!

if selected_town_name:
    print(f"Creating map for: {selected_town_name}")
    print("=" * 70)
    town_map = create_town_map()
    
    if town_map:
        print("\n🗺️  Interactive map created!")
        print("\nMap Features:")
        print("  • Multiple basemaps available (toggle in layer control)")
        print("  • Bedrock Geology layer from VT ANR (semi-transparent)")
        print("  • Town boundary shown in red")
        print("  • Red marker at town center")
        print("  • Use layer control (top right) to toggle layers on/off")
        print("  • Zoom and pan to explore")
        print("=" * 70)
        
        # Display the map
        display(town_map)
else:
    print("⚠️  Please select a town from the dropdown above first!")